# XGBoost Training for Credit Risk

Trains an XGBoost classifier on the processed credit risk data and evaluates it against the existing MLP baseline.

In [22]:
# !pip install xgboost
import xgboost as xgb

In [23]:
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from xgboost import XGBClassifier

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

TRAIN_FILE = DATA_DIR / "train.csv"
VAL_FILE = DATA_DIR / "val.csv"
TEST_FILE = DATA_DIR / "test.csv"

MODEL_FILE = MODELS_DIR / "xgboost_model.json"
CALIBRATOR_FILE = MODELS_DIR / "xgboost_calibrator.pkl"
PREDICTIONS_FILE = RESULTS_DIR / "xgboost_predictions.csv"
METRICS_FILE = RESULTS_DIR / "xgboost_metrics.json"

TARGET_COL = "status"
RANDOM_STATE = 42

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)

In [24]:
# Load data
train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)
test_df = pd.read_csv(TEST_FILE)

X_train = train_df.drop(columns=[TARGET_COL]).values
y_train = train_df[TARGET_COL].values

X_val = val_df.drop(columns=[TARGET_COL]).values
y_val = val_df[TARGET_COL].values

X_test = test_df.drop(columns=[TARGET_COL]).values
y_test = test_df[TARGET_COL].values

# Compute scale_pos_weight to handle class imbalance without oversampling
scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")
print(f"Class balance (train): {y_train.mean():.1%} default rate")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

Training set: (118936, 56)
Validation set: (14867, 56)
Test set: (14867, 56)
Class balance (train): 24.6% default rate
scale_pos_weight: 3.0577


In [25]:
def evaluate_split(name: str, y_true: np.ndarray, proba: np.ndarray) -> dict:
    preds = (proba >= 0.5).astype(int)
    auc_roc = roc_auc_score(y_true, proba)
    auc_pr = average_precision_score(y_true, proba)
    brier = brier_score_loss(y_true, proba)
    positive_rate = y_true.mean()

    print(f"{name} PERFORMANCE (Uncalibrated)")
    print("=" * 80)
    print(f"AUC-ROC:     {auc_roc:.4f}")
    print(f"AUC-PR:      {auc_pr:.4f}")
    print(f"Brier Score: {brier:.4f}")
    print(f"Positives:   {positive_rate:.2%}")
    print("Classification Report:")
    print(classification_report(y_true, preds, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, preds))

    return {
        'dataset': name,
        'auc_roc': float(auc_roc),
        'auc_pr': float(auc_pr),
        'brier_score': float(brier)
    }

In [26]:
# Define and train XGBoost model
xgb_params = {
    'n_estimators': 500,
    'learning_rate': 0.05,
    'max_depth': 5,
    'min_child_weight': 1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.0,
    'reg_lambda': 1.0,
    'reg_alpha': 0.0,
    'scale_pos_weight': scale_pos_weight,
    'early_stopping_rounds': 30,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'random_state': RANDOM_STATE,
    'n_jobs': os.cpu_count()
}

model = XGBClassifier(**xgb_params)

print("Training XGBoost model...")
model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

best_iteration = getattr(model, 'best_iteration', xgb_params['n_estimators'])
print(f"Best iteration: {best_iteration}")

Training XGBoost model...
[0]	validation_0-auc:0.81860
[50]	validation_0-auc:0.87136
[100]	validation_0-auc:0.88044
[150]	validation_0-auc:0.88552
[200]	validation_0-auc:0.88746
[250]	validation_0-auc:0.88895
[300]	validation_0-auc:0.88979
[350]	validation_0-auc:0.89056
[400]	validation_0-auc:0.89108
[450]	validation_0-auc:0.89126
[499]	validation_0-auc:0.89171
Best iteration: 499


In [27]:
# Evaluate on validation and test sets
val_proba = model.predict_proba(X_val)[:, 1]
test_proba = model.predict_proba(X_test)[:, 1]

val_metrics = evaluate_split("Validation", y_val, val_proba)
test_metrics = evaluate_split("Test", y_test, test_proba)

Validation PERFORMANCE (Uncalibrated)
AUC-ROC:     0.8917
AUC-PR:      0.8394
Brier Score: 0.1049
Positives:   24.65%
Classification Report:
              precision    recall  f1-score   support

           0     0.9113    0.9214    0.9163     11203
           1     0.7513    0.7257    0.7383      3664

    accuracy                         0.8732     14867
   macro avg     0.8313    0.8236    0.8273     14867
weighted avg     0.8719    0.8732    0.8725     14867

Confusion Matrix:
[[10323   880]
 [ 1005  2659]]
Test PERFORMANCE (Uncalibrated)
AUC-ROC:     0.9009
AUC-PR:      0.8502
Brier Score: 0.1017
Positives:   24.65%
Classification Report:
              precision    recall  f1-score   support

           0     0.9163    0.9252    0.9207     11203
           1     0.7643    0.7415    0.7527      3664

    accuracy                         0.8799     14867
   macro avg     0.8403    0.8334    0.8367     14867
weighted avg     0.8788    0.8799    0.8793     14867

Confusion Matrix:
[[1

In [28]:
# Platt Scaling: fit a logistic regressor on validation set predictions to correct the
# upward bias that scale_pos_weight introduces into the raw output probabilities.
# The training data distribution is real (24.6% default), so this corrects a loss
# reweighting artifact, not an artificial data shift.
#
# Previous removal rationale was incorrect: see MLP training notebook for full explanation.

calibrator = LogisticRegression()
calibrator.fit(val_proba.reshape(-1, 1), y_val)

val_proba_cal = calibrator.predict_proba(val_proba.reshape(-1, 1))[:, 1]
test_proba_cal = calibrator.predict_proba(test_proba.reshape(-1, 1))[:, 1]

val_brier_cal = brier_score_loss(y_val, val_proba_cal)
test_brier_cal = brier_score_loss(y_test, test_proba_cal)

val_cal_gap_uncal = abs(y_val.mean() - val_proba.mean())
val_cal_gap_cal = abs(y_val.mean() - val_proba_cal.mean())
test_cal_gap_uncal = abs(y_test.mean() - test_proba.mean())
test_cal_gap_cal = abs(y_test.mean() - test_proba_cal.mean())

print("=" * 80)
print("CALIBRATION ANALYSIS")
print("=" * 80)
print(f"\nValidation Set:")
print(f"  Actual default rate:            {y_val.mean():.1%}")
print(f"  Mean predicted (uncalibrated):  {val_proba.mean():.1%}")
print(f"  Mean predicted (calibrated):    {val_proba_cal.mean():.1%}")
print(f"  Calibration gap (uncalibrated): {val_cal_gap_uncal:.4f}")
print(f"  Calibration gap (calibrated):   {val_cal_gap_cal:.4f}")
print(f"  Brier Score (uncalibrated):     {val_metrics['brier_score']:.4f}")
print(f"  Brier Score (calibrated):       {val_brier_cal:.4f}")
print(f"\nTest Set:")
print(f"  Actual default rate:            {y_test.mean():.1%}")
print(f"  Mean predicted (uncalibrated):  {test_proba.mean():.1%}")
print(f"  Mean predicted (calibrated):    {test_proba_cal.mean():.1%}")
print(f"  Calibration gap (uncalibrated): {test_cal_gap_uncal:.4f}")
print(f"  Calibration gap (calibrated):   {test_cal_gap_cal:.4f}")
print(f"  Brier Score (uncalibrated):     {test_metrics['brier_score']:.4f}")
print(f"  Brier Score (calibrated):       {test_brier_cal:.4f}")
print("=" * 80)

CALIBRATION ANALYSIS

Validation Set:
  Actual default rate:            24.6%
  Mean predicted (uncalibrated):  36.8%
  Mean predicted (calibrated):    24.6%
  Calibration gap (uncalibrated): 0.1219
  Calibration gap (calibrated):   0.0000
  Brier Score (uncalibrated):     0.1049
  Brier Score (calibrated):       0.0864

Test Set:
  Actual default rate:            24.6%
  Mean predicted (uncalibrated):  37.0%
  Mean predicted (calibrated):    24.9%
  Calibration gap (uncalibrated): 0.1240
  Calibration gap (calibrated):   0.0022
  Brier Score (uncalibrated):     0.1017
  Brier Score (calibrated):       0.0823


In [29]:
# Save model, calibrator, predictions, and metrics
model.save_model(MODEL_FILE)
print(f"Model saved to {MODEL_FILE}")

joblib.dump(calibrator, CALIBRATOR_FILE)
print(f"Calibrator saved to {CALIBRATOR_FILE}")

test_pred_cal = (test_proba_cal >= 0.5).astype(int)
predictions_df = pd.DataFrame({
    'true_label': y_test,
    'predicted_probability': test_proba_cal,
    'predicted_label': test_pred_cal,
})
predictions_df.to_csv(PREDICTIONS_FILE, index=False)
print(f"Predictions saved to {PREDICTIONS_FILE}")

metrics = {
    'model': 'XGBoost',
    'hyperparameters': {**xgb_params, 'best_iteration': int(best_iteration)},
    'calibration': 'Platt Scaling on validation set',
    'validation_metrics': {
        **val_metrics,
        'brier_score':     float(val_brier_cal),
        'calibration_gap': float(val_cal_gap_cal),
    },
    'test_metrics': {
        **test_metrics,
        'brier_score':     float(test_brier_cal),
        'calibration_gap': float(test_cal_gap_cal),
    }
}

with open(METRICS_FILE, 'w') as f:
    json.dump(metrics, f, indent=4)
print(f"Metrics saved to {METRICS_FILE}")

print("=" * 80)
print("XGBoost training complete")
print("=" * 80)

Model saved to /Users/SanjanaKSL/Desktop/Docs/projects/credit-risk-counterfactual/models/xgboost_model.json
Calibrator saved to /Users/SanjanaKSL/Desktop/Docs/projects/credit-risk-counterfactual/models/xgboost_calibrator.pkl
Predictions saved to /Users/SanjanaKSL/Desktop/Docs/projects/credit-risk-counterfactual/results/xgboost_predictions.csv
Metrics saved to /Users/SanjanaKSL/Desktop/Docs/projects/credit-risk-counterfactual/results/xgboost_metrics.json
XGBoost training complete
